# 04: Sequential model study (STGNN) 

### Goals

Test a new strategy for the classification problem currently under development. 

The strategy that got better results so far was a combination of `Spatial Graph Neural Networks (GNN)` + `Sequential Features`. Now, my goal is to prove that by integrating a `Long Short-Term Memory (LSTM)` algorithm to explicitly process the chronological order of the data, and by adding new physical/operational features, the model will learn both spatial topolofy and temporal dynamics, improving the overall anomaly detection metrics.

### Challenges

- Get to know new concepts and contents regarding `Recurrent Neural Networks (RNNs)`.
- Handle the combination of static physical features (like pipe diameter) with dynamic temporal features (like hourly demand and pressure errors).


### Stack

The study is developed using `PyTorch` and `PyTorch Geometric`. 
Data is manipulated with `Pandas` and `NumPy`.
I also use `Scikit-learn` for evaluation metrics.

# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import wntr
import wntr.graphics as wntr_graphics
import networkx as nx
import os
import torch
import torch.nn as nn
import torch.nn.functional

from torch_geometric.data import Data
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.metrics import classification_report, confusion_matrix

from data_use import (
    normalizar_residuo, taxa_exceedance, calcular_residuo
)

from data_access import (
    get_all, leak_info
)

from data_menagement import (
    process_scenario_data
)

# Prep G

In [7]:
def map_graph(scenarios):
    # carregar topologia
    wn = wntr.network.WaterNetworkModel(scenarios[118]['inp'])
    G = wn.to_graph().to_undirected()

    # lista em ordem fixa de nós
    list_nodes = sorted(wn.junction_name_list, key=int)
    nodes_index = {node: i for i, node in enumerate(list_nodes)} # enumerate devolve par (posição, valor) pra cada elemento de uma lista

    print(list_nodes)
    print(nodes_index)

    return G, list_nodes, nodes_index

In [8]:
def build_edge(G, nodes_index):
    # construir edge_index bidirecional
    start_edge = []
    end_edge = []

    for u, v in G.edges(): 
        # pula qualquer aresta que envolva um nó fora do mapeamento (reservatório)
        if u not in nodes_index or v not in nodes_index:
            continue
        
        idx_u = nodes_index[u]
        idx_v = nodes_index[v]

        start_edge.extend([idx_u, idx_v]) # .extend adiciona múltiplos elementos ao final de uma lista de uma só vez
        end_edge.extend([idx_v, idx_u])

    # cria tensor no formato certo
    edge_index = torch.tensor([start_edge, end_edge], dtype=torch.long)

    return edge_index


# Data preparation

Atualmente, o `data.x` sai no formato 2D `[31, 48]`, enquanto a LSTM precisa entender que isso é uma linha do tempo, e dessa forma não há tal informação. É preciso adicionar uma terceira dimensão no tensor, por isso o formato passa a ser `[31, 48, 1]`, que indica a natureza e o significado dos 48 timesteps: exatamente uma única informação.

No `PyTorch`, é possível mudar isso simplesmente com um reshape no `forward`, sem necessidade de mudar o `DataLoader`.


In [5]:
def build_scenario_data_graph(scenarios, list_nodes, nodes_index, edge_index, janela_dias, window_size, step):
    list_data = []

    for n, item in scenarios.items():
        dfs_nodes, leak_intervals, total_steps = process_scenario_data(item, list_nodes, janela_dias)

        # cada pedaço dão 24 horas de histórico
        for t_init in range(0, total_steps - window_size + 1, step):
            t_end = t_init + window_size

            # dentro de cada janela de 24 horas, cria-se um grafo novo (Data) para a GNN
            # cria a matriz de features x (len com todos nodes da rede) e correspondência y
            n_nodes = len(list_nodes)
            x = torch.zeros((n_nodes, 48), dtype=torch.float)
            y = torch.zeros(n_nodes, dtype=torch.long)

            skip = False

            # for each node, pega a tb tratada (dfs_nodes), cuts 24h from current window e calc mean error_normalizado
            # mean.error_normalizado = feature x[node] naquele grafo
            for node in list_nodes:
                idx = nodes_index[node]
                df_node = dfs_nodes[node]

                # extract slice from error_normalizado nessa window
                slice = df_node['error_normalizado'].iloc[t_init:t_end].values

                # if not len() = 48
                if len(slice) < 48:
                    skip = True
                    break

                # defines the values
                for i, val in enumerate(slice):
                    x[idx, i] = float(val)

                # gabarito
                is_leaking = False
                for leak_node, (start, end) in  leak_intervals.items():
                    # leakege is happening in this window and in this node?
                    if str(node) == str(leak_node) and not (t_end < start or t_init > end):
                        is_leaking = True
                        break

                y[idx] = 1 if is_leaking else 0

        if not skip:
            data = Data(x=x, edge_index=edge_index, y=y)
            list_data.append(data)

    return list_data

In [10]:
scenarios_train = get_all("train")
scenarios_test = get_all("test")
    # path
    # inp
    # info
    # leaks
    # pressures

G, list_nodes, nodes_index = map_graph(scenarios_train)
edge_index = build_edge(G, nodes_index)

list_train = build_scenario_data_graph(scenarios_train, list_nodes, nodes_index, edge_index, janela_dias=14, window_size=48, step=48)
list_val = build_scenario_data_graph(scenarios_test, list_nodes, nodes_index, edge_index, janela_dias=14, window_size=48, step=48)


['2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32']
{'2': 0, '3': 1, '4': 2, '5': 3, '6': 4, '7': 5, '8': 6, '9': 7, '10': 8, '11': 9, '12': 10, '13': 11, '14': 12, '15': 13, '16': 14, '17': 15, '18': 16, '19': 17, '20': 18, '21': 19, '22': 20, '23': 21, '24': 22, '25': 23, '26': 24, '27': 25, '28': 26, '29': 27, '30': 28, '31': 29, '32': 30}


# Train

In [14]:
class GCN_LSTM(torch.nn.Module):
    def __init__(self, lstm_input_size, lstm_hidden_size, gcn_hidden_channels, n_classes):
        super().__init__()

        self.lstm = nn.LSTM(input_size=lstm_input_size,
                            hidden_size=lstm_hidden_size,
                            batch_first=True)
        
        self.conv1 = GCNConv(lstm_hidden_size, gcn_hidden_channels)
        self.conv2 = GCNConv(gcn_hidden_channels, gcn_hidden_channels)
        self.conv3 = GCNConv(gcn_hidden_channels, n_classes)
    
    def forward(self, x, edge_index):
        x_lstm = x.unsqueeze(-1)  # adiciona uma nova dimensão de tamanho 1 no último eixo de um tensor
    
        # passando pela LSTM
        # out -> histório / (h_c, c_n)
        out, (h_n, c_n) = self.lstm(x_lstm)

        # todos os nós, último timestep, todas as features ocultas
        x_relatory = out[:, -1, :]

        # gcn (espaço)
        x_gnn = self.conv1(x_relatory, edge_index)
        x_gnn = F.relu(x_gnn)
        x_gnn = self.conv2(x_gnn, edge_index)
        x_gnn = F.relu(x_gnn)
        x_gnn = self.conv3(x_gnn, edge_index)

        return x_gnn

In [15]:
from torch_geometric.loader import DataLoader

# cria DataLoader
train_loader = DataLoader(list_train, batch_size=32, shuffle=True)
val_loader = DataLoader(list_val, batch_size=32, shuffle=False)

# condig model
modelo = GCN_LSTM(lstm_input_size=1, lstm_hidden_size=32, gcn_hidden_channels=64, n_classes=2)
otimizador = torch.optim.Adam(modelo.parameters(), lr=0.0005)
w_class = torch.tensor([1.0, 150.0])
modelo.train()

for epoca in range(100):
    total_loss = 0

    for batch in train_loader:
        otimizador.zero_grad()                   # zera os gradientes acumulados da rodada anterior

        # PyG groups graphs automatically no objeto 'batch'
        output = modelo(batch.x, batch.edge_index) # faz a previsão (forward pass)

        loss = F.cross_entropy(output, batch.y, weight=w_class)   # calcula o erro
        loss.backward()                          # calcula os gradientes (o quanto cada peso contribuiu pro erro)
        otimizador.step()                        # ajusta os pesos na direção que reduz o erro
        
        total_loss += loss.item()

    if epoca % 10 == 0:
        print(f"Época {epoca}, perda total: {total_loss:.4f}")
        

Época 0, perda total: 0.7073
Época 10, perda total: 0.6228
Época 20, perda total: 0.5482
Época 30, perda total: 0.4637
Época 40, perda total: 0.3455
Época 50, perda total: 0.1958
Época 60, perda total: 0.0511
Época 70, perda total: 0.0022
Época 80, perda total: 0.0001
Época 90, perda total: 0.0000
